In [ ]:
# ========= 0) Imports & Global Config =========
import os, re
import numpy as np
import tensorflow as tf
from itertools import product
from sklearn.metrics import confusion_matrix, roc_auc_score, average_precision_score

print("TensorFlow:", tf.__version__)


### Configurations

In [ ]:
# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Data paths (EDIT if needed)
TRAIN_PATH = r"/kaggle/input/datasets/nadiralii/ac4c-dataset-correct/trainset_labeled.txt"
TEST_PATH  = r"/kaggle/input/datasets/nadiralii/ac4c-dataset-correct/testset_labeled.fasta"
DINUC_PROP_PATH = r"/kaggle/input/datasets/nadiralii/ac4c-dataset-correct/diRNAPhyche.txt"

# Sequence length
L = 201

# -------- Handcrafted controls --------
# PSTNP mode: "ss", "ds", "both"
PSTNP_MODE = "ds" # "both"

# -------- Training controls --------
BATCH_SIZE = 32
EPOCHS_TRAIN = 200

# Model settings
D_MODEL = 128
D_FF = 256
NUM_HEADS = 4
NUM_LAYERS = 2
CNN_FILTERS = (7, 7)
CNN_KERNELS = (3,3)
CNN_DROPOUT = 0.15
TRANSFORMER_DROPOUT = 0.1

# Checkpoint for optional pretraining
OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)
ENCODER_CKPT = os.path.join(OUT_DIR, "pretrained_encoder.weights.h5")


In [ ]:
def read_fasta_labeled(path: str, convert_t_to_u: bool = True):
    """Reads FASTA-like labeled format where the last header token is 0/1 (or label=0/1)."""
    ids, seqs, labels = [], [], []
    cur_id, cur_label, cur_seq = None, None, []

    def flush():
        nonlocal cur_id, cur_label, cur_seq
        if cur_id is not None:
            s = "".join(cur_seq).strip().upper()
            if convert_t_to_u:
                s = s.replace("T", "U")
            ids.append(cur_id)
            seqs.append(s)
            labels.append(int(cur_label))
        cur_id, cur_label, cur_seq = None, None, []

    with open(path, "r", encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            if line.startswith(">"):
                flush()
                header = line[1:]
                m = re.search(r"(?:label=)?([01])\s*$", header)
                if not m:
                    raise ValueError(f"Cannot parse label from header: {line}")
                cur_label = int(m.group(1))
                cur_id = header[: m.start()].strip()
            else:
                cur_seq.append(line)

    flush()
    return ids, seqs, np.array(labels, dtype=np.int64)

def validate_fixed_length(seqs, L_expected=201):
    bad = [(i, len(s)) for i, s in enumerate(seqs) if len(s) != L_expected]
    if bad:
        raise ValueError(f"Found {len(bad)} sequences not length={L_expected}. Example: {bad[:5]}")

tr_ids, tr_seqs, y_train = read_fasta_labeled(TRAIN_PATH)
te_ids, te_seqs, y_test  = read_fasta_labeled(TEST_PATH)
validate_fixed_length(tr_seqs, L)
validate_fixed_length(te_seqs, L)

print("Train:", len(tr_seqs), "Pos:", int(y_train.sum()), "Neg:", int((y_train==0).sum()))
print("Test :", len(te_seqs), "Pos:", int(y_test.sum()), "Neg:", int((y_test==0).sum()))
print("Example id:", tr_ids[0])
print("Example seq:", tr_seqs[0][:30]+"...")

In [ ]:
def all_kmers(k: int, alphabet: str):
    return ["".join(p) for p in product(alphabet, repeat=k)]

class PSTNPComputer:
    """
    PSTNPss/PSTNPds positional encoding.

    - Fits ONLY on training split (train_seqs + y_train)
    - No leave-one-out (because train/test are already separated)
    - transform() uses learned delta tables

    mode:
      ss   -> full RNA alphabet A/C/G/U (64 tri)
      ds   -> degenerate mapping: U->A and G->C; alphabet A/C (8 tri)
      both -> concat [ss, ds]
    """
    def __init__(self, L: int, mode: str = "ss", eps: float = 0.0):
        assert mode in {"ss", "ds", "both"}
        self.L = L
        self.mode = mode
        self.eps = eps
        self._fitted = False

    def _fit_one(self, train_seqs, y_train, alphabet: str, map_fn=None):
        k = 3
        positions = self.L - k + 1
        kmers = all_kmers(k, alphabet)
        kmer_index = {m: i for i, m in enumerate(kmers)}
        V = len(kmers)

        y_train = np.asarray(y_train).astype(int)
        Npos = int((y_train == 1).sum())
        Nneg = int((y_train == 0).sum())
        if Npos == 0 or Nneg == 0:
            raise ValueError("PSTNP fit requires both classes in training split.")

        cpos = np.zeros((positions, V), dtype=np.float64)
        cneg = np.zeros((positions, V), dtype=np.float64)

        for s, lab in zip(train_seqs, y_train):
            s = s.upper().replace("T", "U")
            if map_fn is not None:
                s = map_fn(s)
            for p in range(positions):
                tri = s[p:p+3]
                j = kmer_index.get(tri)
                if j is None:
                    continue
                if lab == 1:
                    cpos[p, j] += 1.0
                else:
                    cneg[p, j] += 1.0

        fpos = (cpos + self.eps) / (Npos + self.eps * V)
        fneg = (cneg + self.eps) / (Nneg + self.eps * V)
        delta = fpos - fneg
        return delta, kmer_index

    def fit(self, train_seqs, y_train):
        # ss tables
        self.delta_ss, self.index_ss = self._fit_one(train_seqs, y_train, alphabet="ACGU", map_fn=None)

        # ds tables: U->A, G->C (RNA equivalent of FileProcessing T->A, G->C)
        def map_ds(s: str) -> str:
            return s.replace("U", "A").replace("G", "C")
        self.delta_ds, self.index_ds = self._fit_one(train_seqs, y_train, alphabet="AC", map_fn=map_ds)

        self._fitted = True
        return self

    def _transform_one(self, seqs, delta, kmer_index, map_fn=None):
        k = 3
        positions = self.L - k + 1
        X = np.zeros((len(seqs), positions), dtype=np.float32)
        for i, s in enumerate(seqs):
            s = s.upper().replace("T", "U")
            if map_fn is not None:
                s = map_fn(s)
            for p in range(positions):
                tri = s[p:p+3]
                j = kmer_index.get(tri)
                X[i, p] = float(delta[p, j]) if j is not None else 0.0
        return X

    def transform(self, seqs):
        if not self._fitted:
            raise RuntimeError("Call fit() before transform().")

        if self.mode == "ss":
            return self._transform_one(seqs, self.delta_ss, self.index_ss, map_fn=None)

        if self.mode == "ds":
            def map_ds(s: str) -> str:
                return s.replace("U", "A").replace("G", "C")
            return self._transform_one(seqs, self.delta_ds, self.index_ds, map_fn=map_ds)

        # both
        def map_ds(s: str) -> str:
            return s.replace("U", "A").replace("G", "C")
        Xss = self._transform_one(seqs, self.delta_ss, self.index_ss, map_fn=None)
        Xds = self._transform_one(seqs, self.delta_ds, self.index_ds, map_fn=map_ds)
        return np.concatenate([Xss, Xds], axis=1)

# -------- Handcrafted controls --------
# PSTNP mode: "ss", "ds", "both"
PSTNP_MODE = "ds"

pst = PSTNPComputer(L=L, mode=PSTNP_MODE).fit(tr_seqs, y_train)
Xtr_pst = pst.transform(tr_seqs)
Xte_pst = pst.transform(te_seqs)
print("PSTNP mode:", PSTNP_MODE)
print("PSTNP shapes:", Xtr_pst.shape, Xte_pst.shape)

In [ ]:
Xtr_hand = Xtr_pst.astype(np.float32)
Xte_hand = Xte_pst.astype(np.float32)
D_HAND = Xtr_hand.shape[1]

print("Handcrafted shapes:", Xtr_hand.shape, Xte_hand.shape)
print("d_hand =", D_HAND)


In [ ]:
NUC_RNA = "ACGU"
NUC2ID = {"A": 0, "C": 1, "G": 2, "U": 3, "T": 3}

def onehot_encode(seq: str, L: int) -> np.ndarray:
    seq = seq.upper().replace("T", "U")
    x = np.zeros((L, 4), dtype=np.float32)
    for i, ch in enumerate(seq[:L]):
        if ch in NUC2ID:
            x[i, NUC2ID[ch]] = 1.0
    return x

def kmer_vocab(k: int, alphabet: str = NUC_RNA):
    return ["".join(p) for p in product(alphabet, repeat=k)]

def kmer_freq_vector(seq: str, k: int, alphabet: str = NUC_RNA) -> np.ndarray:
    """
    Returns normalized k-mer composition vector of length 4^k.
    For fixed RNA alphabet A/C/G/U:
      k=1 -> 4 columns
      k=2 -> 16 columns
      k=3 -> 64 columns
    """
    seq = seq.upper().replace("T", "U")
    vocab = kmer_vocab(k, alphabet)
    idx = {m: i for i, m in enumerate(vocab)}
    x = np.zeros(len(vocab), dtype=np.float32)

    valid_windows = 0
    for i in range(0, len(seq) - k + 1):
        mer = seq[i:i+k]
        if all(ch in alphabet for ch in mer):
            x[idx[mer]] += 1.0
            valid_windows += 1

    if valid_windows > 0:
        x /= float(valid_windows)
    return x

def make_base_inputs(seqs: list[str], L: int):
    n = len(seqs)
    x_onehot = np.zeros((n, L, 4), dtype=np.float32)
    x_k1 = np.zeros((n, 4**1), dtype=np.float32)
    x_k2 = np.zeros((n, 4**2), dtype=np.float32)
    x_k3 = np.zeros((n, 4**3), dtype=np.float32)

    for i, s in enumerate(seqs):
        s = s[:L]
        x_onehot[i] = onehot_encode(s, L)
        x_k1[i] = kmer_freq_vector(s, 1)
        x_k2[i] = kmer_freq_vector(s, 2)
        x_k3[i] = kmer_freq_vector(s, 3)

    return x_onehot, x_k1, x_k2, x_k3

Xtr_onehot, Xtr_k1, Xtr_k2, Xtr_k3 = make_base_inputs(tr_seqs, L)
Xte_onehot, Xte_k1, Xte_k2, Xte_k3 = make_base_inputs(te_seqs, L)

print("onehot:", Xtr_onehot.shape, "k1:", Xtr_k1.shape, "k2:", Xtr_k2.shape, "k3:", Xtr_k3.shape)


In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA, TruncatedSVD

scaler = MinMaxScaler()
Xtr_hand = scaler.fit_transform(Xtr_hand)
Xte_hand = scaler.transform(Xte_hand)


print("After PCA:", Xtr_hand.shape, Xte_hand.shape)

D_HAND = Xtr_hand.shape[1]
print("New d_hand =", D_HAND)


In [ ]:
D_HAND = Xtr_hand.shape[1] # (285,1) #
print("New d_hand =", D_HAND)

In [ ]:
def make_tf_dataset(inputs: dict, y: np.ndarray, batch_size: int, training: bool):
    ds = tf.data.Dataset.from_tensor_slices((inputs, y.astype(np.float32)))
    if training:
        ds = ds.shuffle(min(len(y), 10000), reshuffle_each_iteration=True)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


Xtr_hand = np.array(Xtr_hand).reshape(-1,Xtr_hand.shape[1],1)
Xte_hand = np.array(Xte_hand).reshape(-1,Xtr_hand.shape[1],1)
train_inputs = {"onehot": Xtr_onehot, "k1": Xtr_k1, "k2": Xtr_k2, "k3": Xtr_k3, "hand": Xtr_hand}
test_inputs  = {"onehot": Xte_onehot, "k1": Xte_k1, "k2": Xte_k2, "k3": Xte_k3, "hand": Xte_hand}

ds_tr = make_tf_dataset(train_inputs, y_train, BATCH_SIZE, training=True)
ds_te = make_tf_dataset(test_inputs,  y_test,  BATCH_SIZE, training=False)

batch = next(iter(ds_tr))
print("Batch onehot:", batch[0]["onehot"].shape, "Batch hand:", batch[0]["hand"].shape)

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import confusion_matrix, roc_auc_score, average_precision_score

class PrintOnlyImprovements(tf.keras.callbacks.Callback):
    def __init__(self):
        super().__init__()
        self.best_val_acc = 0.0
        self.best_val_loss = float('inf')

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        val_acc = logs.get('val_accuracy') or logs.get('val_acc')
        val_loss = logs.get('val_loss')
        improved = []
        if val_loss is not None and val_loss < self.best_val_loss:
            self.best_val_loss = val_loss
            improved.append(f"Loss: {val_loss:.4f}")
        if val_acc is not None and val_acc > self.best_val_acc:
            self.best_val_acc = val_acc
            improved.append(f"Acc: {val_acc:.4f}")
        if improved:
            print(f"Epoch {epoch+1}: {' | '.join(improved)}")

np.random.seed(SEED)
tf.random.set_seed(SEED)

# -----------------------------
# Safety: correct dtypes
# -----------------------------
Xtr_onehot = np.asarray(Xtr_onehot, dtype=np.float32)
Xte_onehot = np.asarray(Xte_onehot, dtype=np.float32)

# k-mer frequency vectors, not integer token sequences
Xtr_k1 = np.asarray(Xtr_k1, dtype=np.float32)
Xte_k1 = np.asarray(Xte_k1, dtype=np.float32)
Xtr_k2 = np.asarray(Xtr_k2, dtype=np.float32)
Xte_k2 = np.asarray(Xte_k2, dtype=np.float32)
Xtr_k3 = np.asarray(Xtr_k3, dtype=np.float32)
Xte_k3 = np.asarray(Xte_k3, dtype=np.float32)

Xtr_pst = np.asarray(Xtr_pst, dtype=np.float32)
Xte_pst = np.asarray(Xte_pst, dtype=np.float32)
y_train = np.asarray(y_train).astype(np.float32)
y_test  = np.asarray(y_test).astype(np.float32)

print("Data shapes:")
print("onehot:", Xtr_onehot.shape, Xte_onehot.shape)
print("k1    :", Xtr_k1.shape, Xte_k1.shape, "expected dim=4")
print("k2    :", Xtr_k2.shape, Xte_k2.shape, "expected dim=16")
print("k3    :", Xtr_k3.shape, Xte_k3.shape, "expected dim=64")
print("pstnp :", Xtr_pst.shape, Xte_pst.shape)


def compute_binary_metrics(y_true, y_prob, thr=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float).ravel()
    y_pred = (y_prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    acc = (tp + tn) / max(tp + tn + fp + fn, 1)
    sn = tp / max(tp + fn, 1)
    sp = tn / max(tn + fp, 1)
    denom = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = ((tp * tn) - (fp * fn)) / denom if denom > 0 else 0.0
    roc = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else float("nan")
    ap  = average_precision_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else float("nan")
    return {"ACC": acc, "SN": sn, "SP": sp, "MCC": mcc, "ROC_AUC": roc, "PR_AUC": ap}


def build_cnn_encoder(d_model=32, filters=32, kernel_size=3, dropout=0.15, name="cnn_encoder_downstream"):
    inp = tf.keras.Input(shape=(None, d_model), name=f"{name}_in")
    x = tf.keras.layers.Conv1D(filters, kernel_size, padding="same", activation="relu",
                               kernel_regularizer=tf.keras.regularizers.l2(0.0001))(inp)
    x = tf.keras.layers.AveragePooling1D(2)(x)
    x = tf.keras.layers.Dropout(dropout)(x)
    x = tf.keras.layers.Dense(d_model)(x)
    return tf.keras.Model(inp, x, name=name)

class PositionalEmbedding(tf.keras.layers.Layer):
    def __init__(self, max_len: int, d_model: int, **kwargs):
        super().__init__(**kwargs)
        self.pos_emb = tf.keras.layers.Embedding(input_dim=max_len, output_dim=d_model)
    def call(self, x):
        T = tf.shape(x)[1]
        positions = tf.range(start=0, limit=T, delta=1)
        return x + self.pos_emb(positions)

class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1, **kwargs):
        super().__init__(**kwargs)
        self.mha = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model // num_heads, dropout=dropout)
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(d_ff, activation="relu"),
            tf.keras.layers.Dropout(dropout),
            tf.keras.layers.Dense(d_model),
        ])
        self.ln1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.ln2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.do1 = tf.keras.layers.Dropout(dropout)
        self.do2 = tf.keras.layers.Dropout(dropout)
    def call(self, x, training=True):
        attn_out = self.mha(x, x, training=training)
        x = self.ln1(x + self.do1(attn_out, training=training))
        ffn_out = self.ffn(x, training=training)
        x = self.ln2(x + self.do2(ffn_out, training=training))
        return x


def prepare_final_inputs(feature_names):
    """
    Allowed feature_names:
      onehot, k1, k2, k3, pstnp

    onehot is treated as positional sequence input.
    k1/k2/k3/pstnp are concatenated as selected vector features.
    """
    feature_names = tuple(feature_names)
    Xtr_dict, Xte_dict = {}, {}

    if "onehot" in feature_names:
        Xtr_dict["onehot"] = Xtr_onehot
        Xte_dict["onehot"] = Xte_onehot

    vec_parts_tr, vec_parts_te = [], []
    vector_bank = {
        "k1": (Xtr_k1, Xte_k1),
        "k2": (Xtr_k2, Xte_k2),
        "k3": (Xtr_k3, Xte_k3),
        "pstnp": (Xtr_pst, Xte_pst),
    }

    for name in ["k1", "k2", "k3", "pstnp"]:
        if name in feature_names:
            a_tr, a_te = vector_bank[name]
            vec_parts_tr.append(a_tr)
            vec_parts_te.append(a_te)

    vector_dim = 0
    if vec_parts_tr:
        Xtr_vec = np.concatenate(vec_parts_tr, axis=1).astype(np.float32)
        Xte_vec = np.concatenate(vec_parts_te, axis=1).astype(np.float32)

        scaler = MinMaxScaler()
        Xtr_vec = scaler.fit_transform(Xtr_vec).astype(np.float32)
        Xte_vec = scaler.transform(Xte_vec).astype(np.float32)

        vector_dim = Xtr_vec.shape[1]
        Xtr_dict["hand"] = Xtr_vec.reshape(-1, vector_dim, 1)
        Xte_dict["hand"] = Xte_vec.reshape(-1, vector_dim, 1)

    return Xtr_dict, Xte_dict, vector_dim


def build_final_classifier(feature_names, vector_dim):
    feature_names = tuple(feature_names)
    inputs = {}
    branches = []

    # onehot positional branch
    if "onehot" in feature_names:
        inp_onehot = tf.keras.Input(shape=(L, 4), name="onehot")
        x = tf.keras.layers.Dense(32, activation="relu", name="seq_project")(inp_onehot)
        cnn_encoder = build_cnn_encoder(d_model=32, filters=7, kernel_size=3, dropout=0.15,
                                        name="cnn_encoder_downstream")
        x = cnn_encoder(x)
        x = tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(16, return_sequences=True, dropout=0.2,
                                 recurrent_dropout=0.0, activation="relu")
        )(x)
        x = PositionalEmbedding(max_len=L, d_model=32, name="pos_emb")(x)
        x = TransformerBlock(d_model=32, num_heads=8, d_ff=32, dropout=0.4, name="tr_block_1")(x)
        x = tf.keras.layers.AveragePooling1D(4)(x)
        x = tf.keras.layers.Dropout(0.2)(x)
        x = tf.keras.layers.Flatten()(x)
        inputs["onehot"] = inp_onehot
        branches.append(x)

    # selected vector-feature branch: kmer frequency + PSTNP
    if vector_dim > 0:
        inp_vec = tf.keras.Input(shape=(vector_dim, 1), name="hand")

        x = tf.keras.layers.Dense(32, activation="relu", name="hand_project")(inp_vec)

        cnn_encoder1 = build_cnn_encoder(
            d_model=32,
            filters=7,
            kernel_size=3,
            dropout=0.15,
            name="cnn_encoder"
        )

        x = cnn_encoder1(x)
        x = tf.keras.layers.AveragePooling1D(2)(x)
        x = tf.keras.layers.Dropout(0.3)(x)
        x = tf.keras.layers.Flatten()(x)

        inputs["hand"] = inp_vec
        branches.append(x)

    if not branches:
        raise ValueError("At least one feature must be selected.")
    z = branches[0] if len(branches) == 1 else tf.keras.layers.Concatenate(name="final_fusion")(branches)
    z = tf.keras.layers.Dropout(0.4)(z)
    out = tf.keras.layers.Dense(1, activation="sigmoid")(z)
    return tf.keras.Model(inputs=inputs, outputs=out, name="final_model")


def run_final_experiment(exp_name, feature_names, epochs=200, batch_size=32, lr=0.0005, verbose=1):
    tf.keras.backend.clear_session()
    np.random.seed(SEED)
    tf.random.set_seed(SEED)

    Xtr_dict, Xte_dict, vector_dim = prepare_final_inputs(feature_names)
    print(f"Selected features: {feature_names}; vector_dim={vector_dim}")

    model = build_final_classifier(feature_names=feature_names, vector_dim=vector_dim)
    model.summary()

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=["accuracy"]
    )

    callbacks = [
        PrintOnlyImprovements(),
        tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", mode="max", patience=100, restore_best_weights=True),
    ]

    history = model.fit(
        Xtr_dict,
        y_train,
        validation_data=(Xte_dict, y_test),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=verbose,
        shuffle=True
    )

    yprob = model.predict(Xte_dict, verbose=0).ravel()
    metrics = compute_binary_metrics(y_test, yprob, thr=0.5)
    row = {"Experiment": exp_name, "Features": "+".join(feature_names), **metrics}
    return row, history, model

print("Corrected final model block loaded successfully.")


In [ ]:
row, history, model = run_final_experiment(
    exp_name="Final Model",
    feature_names=("onehot", "k1", "k2", "pstnp"),
    epochs=EPOCHS_TRAIN,
    batch_size=BATCH_SIZE,
    lr=0.0005,
    verbose=1
)

result_df = pd.DataFrame([row])

for col in ["ACC", "SN", "SP", "MCC", "ROC_AUC", "PR_AUC"]:
    result_df[col] = result_df[col].astype(float).round(4)

print("\nStandalone Final Model Result:")
display(result_df)

result_df.to_csv("standalone_final_model_result.csv", index=False)

print("Saved: standalone_final_model_result.csv")

In [ ]:
experiment_list = [
    ("onehot", ("onehot",)),
    ("k1", ("k1",)),
    ("k2", ("k2",)),
    ("k3", ("k3",)),
    ("pstnp", ("pstnp",)),

    ("onehot+pstnp", ("onehot", "pstnp")),
    ("onehot+k1", ("onehot", "k1")),
    ("k1+k2+k3", ("k1", "k2", "k3")),
    ("k1+k2+pstnp", ("k1", "k2", "pstnp")),
    ("k1+k2+k3+pstnp", ("k1", "k2", "k3", "pstnp")),

    ("onehot+k1+pstnp", ("onehot", "k1", "pstnp")),
    ("onehot+k1+k2", ("onehot", "k1", "k2")),
    ("onehot+k1+k2+k3", ("onehot", "k1", "k2", "k3")),
    ("onehot+k1+k2+pstnp", ("onehot", "k1", "k2", "pstnp")),
    ("onehot+k1+k2+k3+pstnp", ("onehot", "k1", "k2", "k3", "pstnp")),
]

# Remove exact duplicates while keeping order
seen = set()
unique_experiments = []
for name, feats in experiment_list:
    key = tuple(feats)
    if key not in seen:
        unique_experiments.append((name, feats))
        seen.add(key)

print("Experiments to run:")
for i, (name, feats) in enumerate(unique_experiments, 1):
    print(f"{i:02d}. {name:35s} -> {feats}")


# ------------------------------------------------------------
# Run all experiments
# ------------------------------------------------------------
final_rows = []
final_histories = {}
final_models = {}

for idx, (exp_name, feature_names) in enumerate(unique_experiments, 1):
    print("" + "=" * 100)
    print(f"Running {idx}/{len(unique_experiments)}: {exp_name}")
    print("=" * 100)

    row, history, model = run_final_experiment(
        exp_name=exp_name,
        feature_names=feature_names,
        epochs=EPOCHS_TRAIN,      
        batch_size=BATCH_SIZE,    
        lr=0.0005,                
        verbose=0
    )

    final_rows.append(row)
    final_histories[exp_name] = history.history
    final_models[exp_name] = model

    print("Completed:", exp_name)
    print(row)


# ------------------------------------------------------------
# Final result table
# ------------------------------------------------------------
final_results_df = pd.DataFrame(final_rows)

metric_cols = ["ACC", "SN", "SP", "MCC", "ROC_AUC", "PR_AUC"]
for col in metric_cols:
    final_results_df[col] = final_results_df[col].astype(float).round(4)

# Journal-style sorting
final_results_df = final_results_df.sort_values(
    by=["ROC_AUC", "PR_AUC", "ACC", "MCC"],
    ascending=False
).reset_index(drop=True)

print("Final Model Results Table:")
display(final_results_df)

# Optional save
final_results_df.to_csv("final_model_independent_results.csv", index=False)
print("Saved file: final_model_independent_results.csv")
